<a href="https://colab.research.google.com/github/Ramyapotnuru26/GenAItasks/blob/main/week2tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Day1**

In [ ]:
!pip install -q sentence-transformers faiss-cpu chromadb

In [ ]:
passages = [
    "Wikipedia: The Taj Mahal is an ivory-white marble mausoleum on the right bank of the river Yamuna in Agra, India.",
    "Wikipedia: Bengaluru is the capital city of Karnataka and is known as India's technology hub.",
    "Python docs: Python is an interpreted, high-level, general-purpose programming language.",
    "NumPy docs: NumPy is a library for numerical computing in Python, offering array objects and linear algebra tools.",
    "PyTorch docs: PyTorch is an open-source machine learning framework used for deep learning and tensor computation.",
    "FAISS docs: FAISS is a library for efficient similarity search and clustering of dense vectors.",
    "Chroma docs: Chroma is an open-source embedding database for AI applications.",
    "Sentence Transformers docs: Sentence Transformers creates dense vector representations for sentences and paragraphs."
]

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = model.encode_document(passages, convert_to_numpy=True, normalize_embeddings=True)

print("Embeddings shape:", doc_embeddings.shape)
print("dtype:", doc_embeddings.dtype)

In [ ]:
import faiss

dimension = doc_embeddings.shape[1]


index = faiss.IndexFlatIP(dimension)

index.add(doc_embeddings.astype("float32"))

query = input("enter a question:")
query_embedding = model.encode_query([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")

k = 3
scores, indices = index.search(query_embedding, k)

print("Query:", query)
print("\nTop results from FAISS:\n")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    print(f"{rank}. score={score:.4f}")
    print(passages[idx])
    print()

**Day 2**

In [ ]:

!pip -q install pypdf sentence-transformers faiss-cpu nltk

import re
import faiss
import nltk
from pypdf import PdfReader
from google.colab import files
from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize


nltk.download("punkt")
nltk.download("punkt_tab")


uploaded = files.upload()
pdf_file = list(uploaded.keys())[0]


reader = PdfReader(pdf_file)
pages = [page.extract_text() for page in reader.pages if page.extract_text()]
text = "\n".join(pages)


def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)   # remove extra spaces
    return text.strip()

text = clean_text(text)


def chunk_text(text, chunk_size=300):
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_len = 0

    for sent in sentences:
        sent_len = len(sent.split())

        if current_len + sent_len <= chunk_size:
            current_chunk.append(sent)
            current_len += sent_len
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sent]
            current_len = sent_len

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

chunks = chunk_text(text, chunk_size=300)
print("Total chunks:", len(chunks))


embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)


index = faiss.IndexFlatIP(embeddings.shape[1])   # cosine similarity
index.add(embeddings)

def retrieve(query, top_k=3):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(q_emb, top_k)
    return [chunks[i] for i in indices[0]]


while True:
    query = input("\nEnter your question (type 'exit' to stop): ")

    if query.lower() == "exit":
        print("Exiting...")
        break

    results = retrieve(query, top_k=3)

    for i, r in enumerate(results, 1):
        print(f"\nPassage {i}:\n{r}\n")

**Day 3**

In [ ]:

!pip -q install sentence-transformers faiss-cpu transformers accelerate

import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

documents = [
    "The capital of France is Paris.",
    "The Great Wall of China is located in China.",
    "Water boils at 100 degrees Celsius at sea level.",
    "The Earth revolves around the Sun.",
    "Python is a popular programming language created by Guido van Rossum.",
    "The largest planet in the solar system is Jupiter.",
    "The Taj Mahal is in Agra, India.",
    "The human heart has four chambers.",
    "Mount Everest is the highest mountain above sea level.",
    "The Pacific Ocean is the largest ocean on Earth."
]


chunks = documents

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedder.encode(chunks, convert_to_numpy=True)


dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)


def retrieve_top_chunks(question, top_k=3):
    q_emb = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_emb, top_k)
    return [chunks[i] for i in indices[0]]


tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def answer_question(question, top_k=3):
    retrieved = retrieve_top_chunks(question, top_k=top_k)
    context = " ".join(retrieved)

    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:
"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=50)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {
        "question": question,
        "retrieved_chunks": retrieved,
        "answer": answer
    }


test_questions = [
    "What is the capital of France?",
    "Who created Python?",
    "What is the largest planet in the solar system?",
    "How many chambers does the human heart have?",
    "Where is the Taj Mahal located?"
]

for i, q in enumerate(test_questions, 1):
    output = answer_question(q, top_k=1)
    print(f"\nTest {i}")
    print("Question:", output["question"])
    print("Retrieved Chunks:", output["retrieved_chunks"])
    print("Answer:", output["answer"])

**Day 4**

In [ ]:
!pip -q install sentence-transformers transformers rank-bm25

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rank_bm25 import BM25Okapi
import numpy as np
import torch
import re

docs = [
    "Python was created by Guido van Rossum.",
    "The chemical formula of water is H2O.",
    "Mount Everest is the highest mountain above sea level.",
    "The human heart has four chambers.",
    "The blue whale is the largest animal on Earth.",
    "The capital of France is Paris.",
    "The Sun is a star at the center of the Solar System.",
    "The Moon is Earth's natural satellite.",
    "The Great Wall is in China.",
    "Bats are mammals, not birds."
]

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

device = "cuda" if torch.cuda.is_available() else "cpu"
llm = llm.to(device)


doc_emb = embed_model.encode(
    docs,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

tokenized_docs = [tokenize(doc) for doc in docs]
bm25 = BM25Okapi(tokenized_docs)


def retrieve_hybrid(query, top_k=3, alpha=0.7, beta=0.3):
    q_emb = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")[0]

    vec_scores = np.dot(doc_emb, q_emb)   # raw vector scores
    bm25_scores = np.array(bm25.get_scores(tokenize(query)), dtype=np.float32)  # raw bm25 scores

    results = []
    for i, doc in enumerate(docs):
        hybrid_score = alpha * float(vec_scores[i]) + beta * float(bm25_scores[i])
        results.append({
            "doc": doc,
            "vector_score": float(vec_scores[i]),
            "bm25_score": float(bm25_scores[i]),
            "hybrid_score": float(hybrid_score)
        })

    results.sort(key=lambda x: x["hybrid_score"], reverse=True)
    return results[:top_k]


def generate_answer(question, passages):
    context = " ".join(passages)
    prompt = f"""Answer using only the context.
If the answer is not in the context, say: I don't know.

Context: {context}
Question: {question}
Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = llm.generate(**inputs, max_new_tokens=30)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def ask(question, top_k=3):
    print("=" * 100)
    print("QUESTION:", question)
    print("\n--- HYBRID RETRIEVAL (RAW VECTOR + RAW BM25) ---")

    results = retrieve_hybrid(question, top_k=top_k)

    for r in results:
        print(f"Doc: {r['doc']}")
        print(f"Vector Score: {r['vector_score']:.4f}")
        print(f"BM25 Score  : {r['bm25_score']:.4f}")
        print(f"Hybrid Score: {r['hybrid_score']:.4f}\n")

    answer = generate_answer(question, [r["doc"] for r in results])
    print("Final Answer:", answer)


questions = [
    "Who created Python?",
    "What is the formula of water?",
    "Which animal is the largest on Earth?",
    "What is the highest mountain above sea level?",
    "What is the capital of France?"
]

for q in questions:
    ask(q)

**Day 5**

In [ ]:
!pip -q install openai faiss-cpu pypdf numpy gradio


import os
import numpy as np
import faiss
import gradio as gr
from pypdf import PdfReader
from google.colab import files
from openai import OpenAI
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("enter a api key:")
client = OpenAI()


uploaded = files.upload()


def read_pdfs(file_names):
    text = ""
    for file_name in file_names:
        reader = PdfReader(file_name)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

raw_text = read_pdfs(uploaded.keys())


def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return [c.strip() for c in chunks if c.strip()]

chunks = chunk_text(raw_text)


def get_embeddings(text_list, model="text-embedding-3-small"):
    resp = client.embeddings.create(
        model=model,
        input=text_list
    )
    return [d.embedding for d in resp.data]

embeddings = get_embeddings(chunks)

dim = len(embeddings[0])
index = faiss.IndexFlatL2(dim)
index.add(np.array(embeddings, dtype="float32"))


def ask_pdf(question, top_k=3, distance_threshold=1.2):
    if not question.strip():
        return "Please enter a question."

    q_emb = client.embeddings.create(
        model="text-embedding-3-small",
        input=[question]
    ).data[0].embedding

    D, I = index.search(np.array([q_emb], dtype="float32"), top_k)

    if D[0][0] > distance_threshold:
        return "No content found"

    context = "\n\n".join([chunks[i] for i in I[0]])

    prompt = f"""
You must answer ONLY from the CONTEXT below.
If the answer is not clearly present in the CONTEXT, reply exactly:
No content found

CONTEXT:
{context}

QUESTION:
{question}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "Answer only from the given PDF context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    answer = response.choices[0].message.content.strip()
    return answer if answer else "No content found"


def clear_all():
    return "", ""

def chat_fn(user_msg, history):
    if not user_msg.strip():
        return "", history

    bot_reply = ask_pdf(user_msg)

    history.append((user_msg, bot_reply))
    return "", history

def clear_chat():
    return []

with gr.Blocks() as demo:
    gr.Markdown("## Gradio Chat Assistant")

    chatbot = gr.Chatbot(label="Chat")

    msg = gr.Textbox(
        placeholder="Ask a question from PDF...",
        show_label=False
    )

    with gr.Row():
        send_btn = gr.Button("Send")
        clear_btn = gr.Button("Clear")

    send_btn.click(chat_fn, inputs=[msg, chatbot], outputs=[msg, chatbot])
    msg.submit(chat_fn, inputs=[msg, chatbot], outputs=[msg, chatbot])

    clear_btn.click(clear_chat, outputs=chatbot)

demo.launch(share=True)